In [ ]:
import pickle
from pathlib import Path

import getdist as gd
import getdist.plots as gdplots
import matplotlib.pyplot as plt
import numpy as np

from expconfig import ExpConfig
from sampling.priors import CompoundPrior

In [ ]:
OUTPUTS_PATH = Path().resolve().parent / "outputs"


def get_results_dir(n: int = 0) -> Path:
    """Get the most recent results directory.

    Parameters
    ----------
    n : int, optional
        Index of the results directory to retrieve.
        0 for the most recent, 1 for the second most recent, etc.
        Default is 0.

    Returns
    -------
    Path
        Path to the results directory.
    """
    output_dirs = sorted(
        OUTPUTS_PATH.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True
    )
    return output_dirs[n]

In [ ]:
RESULTS_DIR = get_results_dir(0)

cfg = ExpConfig.load(RESULTS_DIR / "config.yaml")
prior = CompoundPrior.from_dict(cfg.priors.model_dump())
prior_samples = prior.sample(500_000, np.random.default_rng(1234))

In [ ]:
ln_prob = pickle.load(open(RESULTS_DIR / "lnprob_full.pkl", "rb")).reshape(
    -1, cfg.sampling.nwalkers
)
samples = pickle.load(open(RESULTS_DIR / "samples_full.pkl", "rb")).reshape(
    -1, cfg.sampling.nwalkers, prior.n
)
fig, axes = plt.subplots(1, 1, figsize=(10, 5))
axes.plot(ln_prob, lw=1)
plt.show()

In [ ]:
keep_mask = (ln_prob > -7000).all(axis=0)
ln_prob = ln_prob[75:, keep_mask]
samples = samples[75:, keep_mask, :]

with open(RESULTS_DIR / "lnprob.pkl", "wb") as f:
    pickle.dump(ln_prob.reshape(-1), f)
with open(RESULTS_DIR / "samples.pkl", "wb") as f:
    pickle.dump(samples.reshape(-1, prior.n), f)

fig, axes = plt.subplots(1, 1, figsize=(10, 5))
axes.plot(ln_prob, lw=1)
plt.show()

In [ ]:
names = [
    "dA1",
    "dC1-dA1",
    "dF1-dA1+2dN1",
    "dL1",
    "eta11",
    "eta21",
    "dA2",
    "dC2-dA2",
    "dF2-dA2+2dN2",
    "dL2",
    "eta12",
    "eta22",
]

posterior_samples_gd = gd.MCSamples(
    samples=samples.reshape(-1, prior.n), names=names, labels=names
)
prior_samples_gd = gd.MCSamples(samples=prior_samples, names=names, labels=names)
g = gdplots.getSubplotPlotter()
g.triangle_plot(
    [prior_samples_gd, posterior_samples_gd],
    legend_labels=["Prior", "Posterior"],
    filled=True,
)
g.export(str(RESULTS_DIR / "corner_plot_prior_posterior.png"))

In [ ]:
g.triangle_plot(
    [posterior_samples_gd],
    legend_labels=["Posterior"],
    filled=True,
)
g.export(str(RESULTS_DIR / "corner_plot_posterior.png"))

In [ ]:
from sddr.sddr import fit_marginalised_posterior

model = fit_marginalised_posterior(
    posterior_samples_gd.samples,
    np.arange(prior.n),
    cfg.flow,
    cfg.training,
)

In [ ]:
posterior_samples = model.sample(10_000)
# should check that this looks like the marginals above. If I do stick with looking at posterior predictive distributions then should probably do this straight away and then plot the marginals based on these samples?

In [ ]:
import pandas as pd

from icprem import PREM_IC_RHO, PREM_IC_VP
from tti.traveltimes.paths import calculate_path_direction_vector

noise_levels: dict[str, float] = {
    "ab": 0.95,
    "bc": 0.63,
    "cd": 0.29,
    "df": 0.95,
}

DATA_FILE = Path().resolve().parent / "data" / "brett2024_ic_traveltimes.parquet"

df = pd.read_parquet(DATA_FILE)
ic_in = np.stack(df.in_location.values)
ic_out = np.stack(df.out_location.values)

region = cfg.geometry.to_composite_region()

normalisation = -1 / (2 * PREM_IC_RHO * (PREM_IC_VP * 1e3) ** 2)

path_directions = calculate_path_direction_vector(ic_in, ic_out)
total_distances = region.ray_distances(ic_in, path_directions)

# Not sure why some rays have zero total distance.  Discard for now
valid_rays = total_distances > 0
ic_in = ic_in[valid_rays]
ic_out = ic_out[valid_rays]
path_directions = path_directions[valid_rays]
segment_distances = region.ray_distances_per_region(ic_in, path_directions)
weights = segment_distances / total_distances[valid_rays, None]

dt_over_t = (df.delta_t / df.inner_core_travel_time).values[valid_rays]
#  The noise levels for each reference phase are given in seconds, so we need to convert them to fractional traveltime perturbations by dividing by the inner core travel time.
# In principle this gives a different sigma for each observation.
sigma = (df["reference_phase"].map(noise_levels) / df["inner_core_travel_time"]).values[
    valid_rays
]

In [ ]:
from tti.traveltimes import TravelTimeCalculator
from tti.traveltimes.parametrisations import NestedNoShearDegreesParametriser

rng = np.random.default_rng(1234)

ttc = TravelTimeCalculator(
    ic_in,
    ic_out,
    normalisation=normalisation,
    weights=weights.T,
    parametriser=NestedNoShearDegreesParametriser(),
)
posterior_predictive = ttc(posterior_samples)
posterior_predictive += rng.normal(
    loc=0.0,
    scale=sigma,
    size=posterior_predictive.shape,
)

In [ ]:
# ensuring that the histogram range is the same for both the posterior predictive and the synthetic data, so that densities are comparable
all_dists = np.vstack([posterior_predictive, dt_over_t[None, :]])
hist_range = (all_dists.min(), all_dists.max())
common_kwargs = {
    "bins": 50,
    "histtype": "step",
    "range": hist_range,
    "density": True,
}
plt.hist(
    posterior_predictive.T,
    **common_kwargs,
    label="Posterior predictive",
    color=np.full(posterior_predictive.shape[0], fill_value="r"),
    alpha=0.5,
)
plt.hist(
    dt_over_t,
    **common_kwargs,
    label="Synthetic data",
    color="k",
)
plt.xlabel("Traveltime perturbation dt/t")
plt.legend()
plt.savefig(RESULTS_DIR / "posterior_predictive.png")